# Automatic Ticket Classification & Topic Modeling\n\n## Problem Statement\nYou need to build a model that is able to classify customer complaints based on the products/services. By doing so, you can segregate these tickets into their relevant categories and, therefore, help in the quick resolution of the issue.\n\nYou will be doing topic modelling on the **.json** data provided by the company. Since this data is not labelled, you need to apply NMF to analyse patterns and classify tickets into the following five clusters based on their products/services:\n- **Credit card / Prepaid card**\n- **Bank account services**\n- **Theft/Dispute reporting**\n- **Mortgages/loans**\n- **Others**\n\nWith the help of topic modelling, you will be able to map each ticket onto its respective department/category. You can then use this data to train any supervised model such as logistic regression, decision tree or random forest. Using this trained model, you can classify any new customer complaint support ticket into its relevant department.

## Pipelines that need to be performed:\n1. Data loading\n2. Text preprocessing\n3. Exploratory data analysis (EDA)\n4. Feature extraction\n5. Topic modelling (NMF)\n6. Model building using supervised learning\n7. Model training and evaluation\n8. Model inference

## Importing the necessary libraries

In [ ]:
import os\nimport json\nimport numpy as np\nimport pandas as pd\nimport re, nltk, spacy, string\nimport en_core_web_sm\nnlp = en_core_web_sm.load()\nimport seaborn as sns\nimport matplotlib.pyplot as plt\n%matplotlib inline\nimport warnings\nwarnings.filterwarnings('ignore')\n\nfrom sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, TfidfTransformer\nfrom sklearn.decomposition import NMF\nfrom sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.naive_bayes import MultinomialNB\nfrom sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay\nfrom tqdm import tqdm\ntqdm.pandas()\n\n# Display settings\npd.set_option('display.max_colwidth', None)\npd.set_option('display.max_columns', None)

## 1. Loading the data\nThe data is in JSON format and we convert it to a pandas dataframe.

In [ ]:
# Load JSON or fallback preprocessed dataset\njson_path = 'complaints-2021-05-14_08_16.json'\nif os.path.exists(json_path):\n    with open(json_path) as f:\n        data = json.load(f)\n    df = pd.json_normalize(data)\n    print('Columns in dataset:', df.columns.values)\n    \n    # Rename columns\n    df.rename(columns={\n        '_index': 'index', '_type': 'type', '_id': 'id', '_score': 'score',\n        '_source.tags': 'tags', '_source.zip_code': 'zip_code', '_source.complaint_id': 'complaint_id',\n        '_source.issue': 'issue', '_source.date_received': 'date_received', '_source.state': 'state',\n        '_source.consumer_disputed': 'consumer_disputed', '_source.product': 'product',\n        '_source.company_response': 'company_response', '_source.company': 'company',\n        '_source.submitted_via': 'submitted_via', '_source.date_sent_to_company': 'date_sent_to_company',\n        '_source.company_public_response': 'company_public_response', '_source.sub_product': 'sub_product',\n        '_source.timely': 'timely', '_source.complaint_what_happened': 'complaint_what_happened',\n        '_source.sub_issue': 'sub_issue', '_source.consumer_consent_provided': 'consumer_consent_provided'\n    }, inplace=True)\n    \n    df['complaint_what_happened'].replace('', np.nan, inplace=True)\n    df.dropna(subset=['complaint_what_happened'], inplace=True)\nelse:\n    print('Using local cleaned tickets data...')\n    df = pd.read_csv('model/output/cleaned_tickets.csv')\n\ndf.head()

## 2. Text Preprocessing\n- Lowercase conversion\n- Remove text in square brackets\n- Remove punctuation\n- Remove words with numbers\n- Lemmatization with stop words removal\n- POS tag filtering for NN nouns only\n- Remove -PRON- and masked personal details (xxxx)

In [ ]:
def clean_text(text):\n    if pd.isna(text): return ''\n    text = str(text).lower()\n    text = re.sub(r'\[.*?\]', ' ', text)\n    text = re.sub(r'[^\w\s]', ' ', text)\n    text = re.sub(r'\w*\d\w*', ' ', text)\n    text = re.sub(r'\s+', ' ', text).strip()\n    return text\n\nstopwords = nlp.Defaults.stop_words\n\ndef lemmatization(texts):\n    lemma_sentences = []\n    for doc in tqdm(nlp.pipe(texts, batch_size=256)):\n        sent = [token.lemma_ for token in doc if token.text not in stopwords and len(token.lemma_) > 1]\n        lemma_sentences.append(' '.join(sent))\n    return lemma_sentences\n\ndef extract_pos_tags(texts):\n    pos_sentences = []\n    for doc in tqdm(nlp.pipe(texts, batch_size=256)):\n        sent = [token.text for token in doc if token.tag_ == 'NN']\n        pos_sentences.append(' '.join(sent))\n    return pos_sentences\n\ndf_clean = pd.DataFrame()\ndf_clean['complaint_what_happened'] = df['complaint_what_happened'].progress_apply(clean_text)\ndf_clean['complaint_what_happened_lemmatized'] = lemmatization(df_clean['complaint_what_happened'])\ndf_clean['complaint_POS_removed'] = extract_pos_tags(df_clean['complaint_what_happened_lemmatized'])\ndf_clean['Complaint_clean'] = df_clean['complaint_POS_removed'].str.replace('-PRON-', '', regex=False)\ndf_clean['Complaint_clean'] = df_clean['Complaint_clean'].str.replace('xxxx', '', regex=False)\ndf_clean.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
def get_top_n_words(corpus, n=1, count=10):\n    vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(corpus)\n    bag_of_words = vec.transform(corpus)\n    sum_words = bag_of_words.sum(axis=0)\n    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]\n    return sorted(words_freq, key=lambda x: x[1], reverse=True)[:count]\n\nprint('Top 10 Unigrams:', get_top_n_words(df_clean['Complaint_clean'], 1, 10))\nprint('Top 10 Bigrams:', get_top_n_words(df_clean['Complaint_clean'], 2, 10))\nprint('Top 10 Trigrams:', get_top_n_words(df_clean['Complaint_clean'], 3, 10))

## 4. Feature Extraction\nConverting cleaned complaints to TF-IDF matrix (max_df=0.98, min_df=2, stop_words='english').

In [ ]:
tf_idf_vec = TfidfVectorizer(max_df=0.98, min_df=2, stop_words='english')\ntfidf = tf_idf_vec.fit_transform(df_clean['Complaint_clean'])\nprint('TF-IDF matrix shape:', tfidf.shape)

## 5. Topic Modelling using NMF (5 Topics)\nDecompose into 5 clusters and map onto products/services:\n- 0: Bank Account services\n- 1: Credit card or prepaid card\n- 2: Others\n- 3: Theft/Dispute Reporting\n- 4: Mortgage/Loan

In [ ]:
num_topics = 5\nnmf_model = NMF(n_components=num_topics, random_state=40)\nnmf_model.fit(tfidf)\n\nfor index, topic in enumerate(nmf_model.components_):\n    print(f'THE TOP 15 WORDS FOR TOPIC #{index}:')\n    print([tf_idf_vec.get_feature_names_out()[i] for i in topic.argsort()[-15:]])\n    print('\n')\n\ntopic_values = nmf_model.transform(tfidf)\ndf_clean['Topic'] = topic_values.argmax(axis=1)\n\nTopic_names = {\n    0: 'Bank Account services',\n    1: 'Credit card or prepaid card',\n    2: 'Others',\n    3: 'Theft/Dispute Reporting',\n    4: 'Mortgage/Loan'\n}\ndf_clean['Topic_category'] = df_clean['Topic'].map(Topic_names)\ndf_clean[['complaint_what_happened', 'Topic', 'Topic_category']].head()

## 6 & 7. Supervised Model Building, Training & Evaluation\nModels evaluated:\n- Logistic Regression\n- Decision Tree Classifier\n- Random Forest Classifier\n- Multinomial Naive Bayes\n- XGBoost Classifier

In [ ]:
training_data = df_clean[['complaint_what_happened', 'Topic']].copy()\n\ncount_vect = CountVectorizer()\nX_train_counts = count_vect.fit_transform(training_data['complaint_what_happened'])\ntfidf_transformer = TfidfTransformer()\nX_train_tf = tfidf_transformer.fit_transform(X_train_counts)\n\ntrain_X, test_X, train_y, test_y = train_test_split(\n    X_train_tf, training_data['Topic'], test_size=0.2, random_state=40, stratify=training_data['Topic']\n)\n\ndef eval_model(y_test, y_pred, y_pred_proba, type_name='Test'):\n    print(f'{type_name} Results:')\n    print('Accuracy: ', round(accuracy_score(y_test, y_pred), 2))\n    print('Precision: ', round(precision_score(y_test, y_pred, average='weighted', zero_division=0), 2))\n    print('Recall: ', round(recall_score(y_test, y_pred, average='weighted', zero_division=0), 2))\n    print('F1 Score: ', round(f1_score(y_test, y_pred, average='weighted', zero_division=0), 2))\n    if y_pred_proba is not None:\n        try:\n            print('ROC AUC Score: ', round(roc_auc_score(y_test, y_pred_proba, average='weighted', multi_class='ovr'), 2))\n        except Exception:\n            pass\n    print('Classification Report:\n', classification_report(y_test, y_pred, zero_division=0))\n\n# Train and evaluate Logistic Regression\nlr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=40, class_weight='balanced')\nlr_model.fit(train_X, train_y)\neval_model(test_y, lr_model.predict(test_X), lr_model.predict_proba(test_X), type_name='Logistic Regression Test')

## 8. Model Inference\nClassify new customer complaints into their relevant department.

In [ ]:
df_complaints = pd.DataFrame({'complaints': [\n    "I can not get from chase who services my mortgage, who owns it and who has original loan docs",\n    "The bill amount of my credit card was debited twice. Please look into the matter and resolve at the earliest.",\n    "I want to open a salary account at your downtown branch. Please provide me the procedure.",\n    "Yesterday, I received a fraudulent email regarding renewal of my services.",\n    "What is the procedure to know my CIBIL score?",\n    "I need to know the number of bank branches and their locations in the city of Dubai"\n]})\n\ndef predict_complaint(text):\n    X_new_counts = count_vect.transform([text])\n    X_new_tfidf = tfidf_transformer.transform(X_new_counts)\n    predicted = lr_model.predict(X_new_tfidf)[0]\n    return Topic_names[predicted]\n\ndf_complaints['tag'] = df_complaints['complaints'].apply(predict_complaint)\ndf_complaints